# SQL & Database Interview Questions & Answers

Companion to the [complete article](https://todatabeyond.com/blog/sql-and-database-interview-questions-and-answers-for-data-scientists). Run all cells in order with Python 3.11+, Jupyter/Colab and SQLite 3.39+ (RIGHT/FULL JOIN support). Setup registers the small %%sql cell magic explicitly. Python standard-library sqlite3 runs actual SQL in isolated in-memory databases. No pip packages beyond the Jupyter runtime, external server, API keys, GPU or dataset downloads are required.

All thirteen age rows are recovered exactly from the original figure. The login and transaction images contain column/type schemas only, not actual data rows; those fixtures, joins and price/country values are synthetic teaching data. Add transaction_id as a stable tie-breaker for equal transaction timestamps. Datetimes are ISO text without timezone offsets, using the displayed calendar dates. No original numerical login/transaction results are claimed.

Recover the original untruncated gist and both SQL code-image examples before maintenance. The reactivation gist has a reversed BETWEEN interval, missing user correlation, missing GROUP BY and closing syntax, and counts logins rather than distinct monthly users. Replace PostgreSQL DATE_TRUNC/INTERVAL with SQLite month-start date expressions and deduplicate user/month pairs before NOT EXISTS. This follows the question's literal definition (active now, absent last month); it also counts first-time users and first observed months. It does NOT add an earlier-history condition. A business definition requiring known prior activity would need an additional EXISTS and adequate history; truncating available history can misclassify newcomers. Report only observed qualifying months, not a zero-filled calendar.

The source says WHERE EXISTS in prose, but absence needs NOT EXISTS. Preserve original prose in the blog; private review records discrepancies. Keep all six distinct original figures, omitting duplicate schema images in the old questions index. The third-transaction code image misspells transaction_date in its final projection; use one consistent spelling and add deterministic ORDER BY transaction_id after date. ROW_NUMBER selects the third row, not the third distinct day.

Key behavior is engine-specific. Enable PRAGMA foreign_keys before writes. Use explicit INT PRIMARY KEY NOT NULL (not SQLite's auto-generated INTEGER PRIMARY KEY rowid alias). SQLite UNIQUE permits multiple NULL values, unlike the source's one-NULL generalization. Foreign keys can target a suitable UNIQUE parent key, not only a primary key. SQL joins preserve multiplicity and do not match NULL with NULL under equality; Venn diagrams alone do not explain that. NoSQL consistency/ACID/scaling/performance claims vary by engine and configuration; the original comparison is preserved, not benchmarked or universally endorsed. Q5 is conceptual Markdown, not an invented executable NoSQL service.

The final cell compares full SQL results with independent Python multiset/date/set/sort expectations, boundary fixtures and 64 seeded cases. Committed counts are null and outputs empty. Original source SQL and revision are retained in a nonexecuting appendix, not presented as passing code.


In [ ]:
import sqlite3
from IPython.core.magic import register_cell_magic

assert sqlite3.sqlite_version_info >= (3, 39, 0), 'RIGHT/FULL JOIN examples require SQLite 3.39+'
connection = sqlite3.connect(':memory:')
connection.execute('PRAGMA foreign_keys = ON')
SCHEMA = """
CREATE TABLE join_left (row_id INTEGER PRIMARY KEY, join_key INTEGER, label TEXT);
CREATE TABLE join_right (row_id INTEGER PRIMARY KEY, join_key INTEGER, label TEXT);
CREATE TABLE sample_data (id INTEGER PRIMARY KEY, age INTEGER, name TEXT);
CREATE TABLE tablename (COLUMN TEXT, price REAL, country TEXT);
CREATE TABLE user_logins (user_id INTEGER NOT NULL, login_date TEXT NOT NULL);
CREATE TABLE transactions (transaction_id INTEGER PRIMARY KEY, user_id INTEGER NOT NULL,
                           spend REAL NOT NULL, transaction_date TEXT NOT NULL);
"""
connection.executescript(SCHEMA)
left_fixture = [(1, 1, 'A1'), (2, 2, 'A2'), (3, 2, 'A3'), (4, None, 'A-null')]
right_fixture = [(10, 2, 'B1'), (11, 2, 'B2'), (12, 3, 'B3'), (13, None, 'B-null')]
connection.executemany('INSERT INTO join_left VALUES (?, ?, ?)', left_fixture)
connection.executemany('INSERT INTO join_right VALUES (?, ?, ?)', right_fixture)
# All thirteen age rows are transcribed from the original source figure.
age_fixture = [(1, 42, 'A'), (2, 25, 'B'), (3, 42, 'C'), (4, 85, 'D'),
               (5, 93, 'E'), (6, 22, 'F'), (7, 40, 'G'), (8, 20, 'H'),
               (9, 77, 'I'), (10, 100, 'J'), (11, 1, 'K'), (12, 110, 'L'), (13, 124, 'M')]
connection.executemany('INSERT INTO sample_data VALUES (?, ?, ?)', age_fixture)
price_fixture = [('A', 9, 'USA'), ('B', 10, 'UK'), ('C', 50, 'USA'),
                 ('D', 100, None), ('E', 101, 'USA'), ('F', None, 'CA')]
connection.executemany('INSERT INTO tablename VALUES (?, ?, ?)', price_fixture)
# The source supplies schemas only for logins and transactions, not input rows.
# These are synthetic teaching fixtures, including repeated logins and tied dates.
login_fixture = [(1, '2022-12-31 23:59:59'), (1, '2023-01-01 00:00:00'),
                 (1, '2023-01-15 12:00:00'), (1, '2023-03-01 00:00:00'),
                 (1, '2023-03-31 23:59:59'), (1, '2023-04-01 00:00:00'),
                 (2, '2023-01-31 23:59:59'), (2, '2023-02-28 23:59:59'),
                 (2, '2023-03-01 00:00:00'), (3, '2023-02-01 00:00:00'),
                 (3, '2023-02-01 00:00:00'), (3, '2023-04-01 00:00:00'),
                 (4, '2023-03-31 12:00:00'), (5, '2023-01-01 00:00:00'),
                 (5, '2023-03-01 00:00:00')]
connection.executemany('INSERT INTO user_logins VALUES (?, ?)', login_fixture)
transaction_fixture = [(1, 1, 10.0, '2023-01-01 10:00:00'),
                       (2, 1, 20.0, '2023-02-01 10:00:00'),
                       (3, 1, 30.0, '2023-03-01 10:00:00'),
                       (4, 1, 40.0, '2023-04-01 10:00:00'),
                       (5, 2, 50.0, '2023-01-02 09:00:00'),
                       (6, 2, 60.0, '2023-01-02 09:00:00'),
                       (7, 2, 70.0, '2023-01-02 09:00:00'),
                       (8, 3, 80.0, '2023-01-03 09:00:00'),
                       (9, 3, 90.0, '2023-01-04 09:00:00')]
connection.executemany('INSERT INTO transactions VALUES (?, ?, ?, ?)', transaction_fixture)
query_results = {}
query_texts = {}

def print_result(columns, rows):
    text_rows = [[str(value) if value is not None else 'NULL' for value in row] for row in rows]
    widths = [max([len(column)] + [len(row[i]) for row in text_rows]) for i, column in enumerate(columns)]
    def format_row(row):
        return ' | '.join(value.ljust(width) for value, width in zip(row, widths)).rstrip()
    print(format_row(columns))
    print('-+-'.join('-' * width for width in widths))
    for row in text_rows:
        print(format_row(row))
    print('Rows:', len(rows))

@register_cell_magic
def sql(line, cell):
    key = line.strip()
    cursor = connection.execute(cell)
    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()
    query_texts[key] = cell.strip()
    query_results[key] = (columns, rows)
    print_result(columns, rows)


print('SQLite:', sqlite3.sqlite_version, '| Foreign keys:', connection.execute('PRAGMA foreign_keys').fetchone()[0])
print('Recovered age rows: 13. Synthetic joins: 4 + 4 rows; prices: 6; logins: 15; transactions: 9.')
print('In-memory SQL magic ready; no external database, API keys, data files or downloads.')


## 1. SQL Join Types


In [ ]:
%%sql inner-join
SELECT l.row_id AS left_row, l.join_key AS left_key, l.label AS left_label,
       r.row_id AS right_row, r.join_key AS right_key, r.label AS right_label
FROM join_left AS l
INNER JOIN join_right AS r ON l.join_key = r.join_key
ORDER BY (l.row_id IS NULL), l.row_id, r.row_id;


In [ ]:
%%sql left-join
SELECT l.row_id AS left_row, l.join_key AS left_key, l.label AS left_label,
       r.row_id AS right_row, r.join_key AS right_key, r.label AS right_label
FROM join_left AS l
LEFT JOIN join_right AS r ON l.join_key = r.join_key
ORDER BY (l.row_id IS NULL), l.row_id, r.row_id;


In [ ]:
%%sql right-join
SELECT l.row_id AS left_row, l.join_key AS left_key, l.label AS left_label,
       r.row_id AS right_row, r.join_key AS right_key, r.label AS right_label
FROM join_left AS l
RIGHT JOIN join_right AS r ON l.join_key = r.join_key
ORDER BY (l.row_id IS NULL), l.row_id, r.row_id;


In [ ]:
%%sql full-join
SELECT l.row_id AS left_row, l.join_key AS left_key, l.label AS left_label,
       r.row_id AS right_row, r.join_key AS right_key, r.label AS right_label
FROM join_left AS l
FULL OUTER JOIN join_right AS r ON l.join_key = r.join_key
ORDER BY (l.row_id IS NULL), l.row_id, r.row_id;


## 2. Primary, Foreign and Unique Keys


In [ ]:
KEY_SCHEMA = """
CREATE TABLE parent_keys (
    id INT PRIMARY KEY NOT NULL,
    email TEXT UNIQUE
);
CREATE TABLE child_keys (
    id INT PRIMARY KEY NOT NULL,
    parent_id INT REFERENCES parent_keys(id)
);
"""
connection.executescript(KEY_SCHEMA)
connection.executemany('INSERT INTO parent_keys VALUES (?, ?)',
                       [(1, 'one@example.test'), (2, None), (3, None)])
connection.executemany('INSERT INTO child_keys VALUES (?, ?)', [(1, 1), (2, None)])
print('Primary, UNIQUE and foreign-key constraints created.')
print('Parents:', connection.execute('SELECT COUNT(*) FROM parent_keys').fetchone()[0])
print('Nullable UNIQUE emails:', connection.execute('SELECT COUNT(*) FROM parent_keys WHERE email IS NULL').fetchone()[0])
print('Children:', connection.execute('SELECT COUNT(*) FROM child_keys').fetchone()[0])


In [ ]:
rejected_writes = [
    ('Duplicate primary key', 'INSERT INTO parent_keys VALUES (?, ?)', (1, 'other@example.test')),
    ('Duplicate unique email', 'INSERT INTO parent_keys VALUES (?, ?)', (4, 'one@example.test')),
    ('Null primary key', 'INSERT INTO parent_keys VALUES (?, ?)', (None, 'null@example.test')),
    ('Missing referenced parent', 'INSERT INTO child_keys VALUES (?, ?)', (3, 999)),
]
for label, statement, parameters in rejected_writes:
    connection.execute('SAVEPOINT rejected_write')
    try:
        connection.execute(statement, parameters)
    except sqlite3.IntegrityError:
        print(label + ': rejected')
    else:
        raise AssertionError(label + ' unexpectedly accepted')
    finally:
        connection.execute('ROLLBACK TO rejected_write')
        connection.execute('RELEASE rejected_write')


## 3. BETWEEN and IN Operators


In [ ]:
%%sql age-between
SELECT * FROM sample_data
WHERE age BETWEEN 20 AND 40
ORDER BY id;


In [ ]:
%%sql age-in
SELECT * FROM sample_data
WHERE age IN (20, 40)
ORDER BY id;


In [ ]:
%%sql price-between
SELECT * FROM tablename
WHERE price BETWEEN 10 AND 100
ORDER BY COLUMN;


In [ ]:
%%sql country-in
SELECT COLUMN FROM tablename
WHERE 'USA' IN (country)
ORDER BY COLUMN;


## 4. Count Reactivated Users


In [ ]:
%%sql reactivation
-- Count distinct users active this month with no login in the previous month.
WITH monthly_logins AS (
    SELECT DISTINCT user_id, date(login_date, 'start of month') AS login_month
    FROM user_logins
)
SELECT current_month.login_month AS current_month,
       COUNT(*) AS num_reactivated_users
FROM monthly_logins AS current_month
WHERE NOT EXISTS (
    SELECT 1
    FROM monthly_logins AS last_month
    WHERE last_month.user_id = current_month.user_id
      AND last_month.login_month = date(current_month.login_month, '-1 month')
)
GROUP BY current_month.login_month
ORDER BY current_month.login_month;


## 5. Relational and NoSQL Databases


Answer:

Advantages of Relational Databases: Ensure data integrity through a defined schema and ACID properties. Easy to get started with and use for small-scale applications. Lends itself well to vertical scaling. Uses an almost standard query language, making learning or switching between types of relational databases easy.

Advantages of NoSQL Databases: Offers more flexibility in data format and representations, which makes working with Unstructured or semistructured data easier. Hence, useful when still the data schema or adding new features/functionality rapidly like in a startup environment to scale with horizontal scaling. Lends itself better to applications that need to be highly available.

Disadvantages of Relational Databases: Data schema needs to be known in advance. Ale schemas is possible, but frequent changes to the schema for large tables can cause performance issues. Horizontal scaling is relatively difficult, leading to eventual performance bottlenecks

Disadvantages of NoSQL Databases: As outlined by the BASE framework, weaker guarantees of data correctness are made due to the soft-state and eventual consistency property. Managing consistency can also be difficult due to the lack of a predefined schema that’s strictly adhered to. Depending on the type of NoSQL database, it can be challenging for the database to handle its types of complex queries or access patterns.


## 6. Find Each User’s Third Transaction


In [ ]:
%%sql third-transaction
WITH nums AS (
    SELECT user_id, spend, transaction_date,
           ROW_NUMBER() OVER (
               PARTITION BY user_id
               ORDER BY transaction_date, transaction_id
           ) AS trans_num
    FROM transactions
)
SELECT user_id, spend, transaction_date
FROM nums
WHERE trans_num = 3
ORDER BY user_id;


## Independent regression checks


In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta
import random

def join_expected(left, right, kind):
    result = []
    for l in left:
        matches = [r for r in right if l[1] is not None and r[1] is not None and l[1] == r[1]]
        result.extend([(*l, *r) for r in matches])
        if not matches and kind in ('left', 'full'):
            result.append((*l, None, None, None))
    if kind in ('right', 'full'):
        for r in right:
            if not any(l[1] is not None and r[1] is not None and l[1] == r[1] for l in left):
                result.append((None, None, None, *r))
    return sorted(result, key=lambda row: (row[0] is None, row[0] or 0, row[3] or 0))

for kind in ('inner', 'left', 'right', 'full'):
    actual = query_results[kind + '-join'][1]
    assert actual == join_expected(left_fixture, right_fixture, kind)
assert [len(query_results[k + '-join'][1]) for k in ('inner', 'left', 'right', 'full')] == [4, 6, 6, 8]
assert connection.execute('PRAGMA foreign_keys').fetchone() == (1,)
assert connection.execute('PRAGMA foreign_key_check').fetchall() == []
assert connection.execute('SELECT * FROM parent_keys ORDER BY id').fetchall() == [(1, 'one@example.test'), (2, None), (3, None)]
assert connection.execute('SELECT * FROM child_keys ORDER BY id').fetchall() == [(1, 1), (2, None)]
assert query_results['age-between'][1] == [(2, 25, 'B'), (6, 22, 'F'), (7, 40, 'G'), (8, 20, 'H')]
assert query_results['age-in'][1] == [(7, 40, 'G'), (8, 20, 'H')]
assert query_results['price-between'][1] == price_fixture[1:4]
assert query_results['country-in'][1] == [('A',), ('C',), ('E',)]

def expected_months(logins):
    active = {(user, datetime.fromisoformat(stamp).date().replace(day=1)) for user, stamp in logins}
    counts = defaultdict(int)
    for user, month in active:
        previous = (month - timedelta(days=1)).replace(day=1)
        if (user, previous) not in active:
            counts[month.isoformat()] += 1
    return sorted(counts.items())

def expected_thirds(transactions):
    groups = defaultdict(list)
    for row in transactions:
        groups[row[1]].append(row)
    result = []
    for user, rows in sorted(groups.items()):
        rows.sort(key=lambda row: (row[3], row[0]))
        if len(rows) >= 3:
            chosen = rows[2]
            result.append((user, chosen[2], chosen[3]))
    return result

assert query_results['reactivation'][1] == expected_months(login_fixture)
assert query_results['reactivation'][1] == [('2022-12-01', 1), ('2023-01-01', 2), ('2023-02-01', 1), ('2023-03-01', 3), ('2023-04-01', 1)]
assert query_results['third-transaction'][1] == expected_thirds(transaction_fixture)
assert query_results['third-transaction'][1] == [(1, 30.0, '2023-03-01 10:00:00'), (2, 70.0, '2023-01-02 09:00:00')]

def check_fixture(left, right, logins, transactions):
    db = sqlite3.connect(':memory:')
    db.executescript(SCHEMA)
    db.executemany('INSERT INTO join_left VALUES (?, ?, ?)', left)
    db.executemany('INSERT INTO join_right VALUES (?, ?, ?)', right)
    db.executemany('INSERT INTO user_logins VALUES (?, ?)', logins)
    db.executemany('INSERT INTO transactions VALUES (?, ?, ?, ?)', transactions)
    for kind in ('inner', 'left', 'right', 'full'):
        assert db.execute(query_texts[kind + '-join']).fetchall() == join_expected(left, right, kind)
    assert db.execute(query_texts['reactivation']).fetchall() == expected_months(logins)
    assert db.execute(query_texts['third-transaction']).fetchall() == expected_thirds(transactions)
    db.close()

check_fixture([], [], [], [])
check_fixture(left_fixture, [], [], [])
check_fixture([], right_fixture, [], [])
check_fixture(list(reversed(left_fixture)), list(reversed(right_fixture)),
              list(reversed(login_fixture)), list(reversed(transaction_fixture)))
boundary_logins = [(1, '2023-12-31 23:59:59'), (1, '2024-01-01 00:00:00'),
                   (1, '2024-03-31 23:59:59'), (2, '2024-02-29 23:59:59'),
                   (2, '2024-03-01 00:00:00'), (3, '2024-01-31 23:59:59'),
                   (3, '2024-03-01 00:00:00'), (3, '2024-03-01 00:00:00')]
check_fixture([], [], boundary_logins, [])
assert expected_months(boundary_logins) == [('2023-12-01', 1), ('2024-01-01', 1), ('2024-02-01', 1), ('2024-03-01', 2)]
rng = random.Random(42)
for case in range(64):
    left = [(i + 1, rng.choice([None, 1, 2, 3]), 'L' + str(i)) for i in range(rng.randrange(7))]
    right = [(i + 10, rng.choice([None, 1, 2, 4]), 'R' + str(i)) for i in range(rng.randrange(7))]
    dates = ['2023-12-31 23:59:59', '2024-01-01 00:00:00', '2024-02-29 23:59:59', '2024-03-01 00:00:00', '2024-05-31 12:00:00']
    logins = [(rng.randrange(1, 6), rng.choice(dates)) for _ in range(rng.randrange(25))]
    transactions = [(i + 1, rng.randrange(1, 6), float(rng.randrange(-10, 101)), rng.choice(dates)) for i in range(rng.randrange(25))]
    rng.shuffle(left); rng.shuffle(right); rng.shuffle(logins); rng.shuffle(transactions)
    check_fixture(left, right, logins, transactions)
connection.close()
print('All 12 executable examples passed; four join outputs match independent multiset expectations (4/6/6/8 rows).')
print('All 13 original age rows and both screenshot result sets match; price/country filters and key constraints pass.')
print('Monthly distinct-user counts: Dec 1, Jan 2, Feb 1, Mar 3, Apr 1; third transactions: users 1 and 2.')
print('Empty/one-sided/reversed inputs, duplicate/null join keys, duplicate logins, year/leap-month boundaries and tied transactions pass.')
print('All 64 seeded fixture sets match independent Python date/set/sort oracles; no external database or files.')


## Recovered source SQL (not executed)

[Original gist](https://gist.github.com/youssefHosni/9e66ee05eea573eff3e088b81e339d90) · revision `9476c680a790335134f8c9b8789bcc6c2c4fd939`

```sql
SELECT 
    DATE_TRUNC('month', current_month.login_date) AS current_month,
    COUNT(*) AS num_reactivated_users 
FROM 
    user_logins current_month
WHERE
    NOT EXISTS (
      SELECT 
        *
      FROM 
        user_logins last_month
      WHERE
        DATE_TRUNC('month', last_month.login_date) BETWEEN DATE_TRUNC('month', current_month.login_date) AND DATE_TRUNC('month', current_month.login_date) - INTERVAL '1 month'
)
```

### Original SQL image transcriptions

```sql
SELECT * FROM sample_data WHERE age BETWEEN 20 AND 40
SELECT * FROM sample_data WHERE age IN (20,40)
```

```sql
WITH nums AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY user_id
               ORDER BY transaction_date
           ) AS trans_num
    FROM transactions
)
SELECT user_id, spend, transcation_date
FROM nums
WHERE trans_num = 3
```
